In [1]:
import json
import numpy as np
import pandas as pd
import pickle5 as pickle
import copy

In [2]:
#pip show pickle5

In [3]:
rdf_all = pd.read_csv('./modcloth/df_electronics.csv', sep=",")
rdf_all['user_id'] = rdf_all['user_id'].astype('category').cat.codes
rdf_all.head()

,item_id,user_id,rating,timestamp,model_attr,category,brand,year,user_attr,split
0,0,0,5.0,1999-06-13,Female,Portable Audio & Video,NaN,1999,NaN,0
1,0,1,5.0,1999-06-14,Female,Portable Audio & Video,NaN,1999,NaN,0
2,0,2,3.0,1999-06-17,Female,Portable Audio & Video,NaN,1999,NaN,0
3,0,3,1.0,1999-07-01,Female,Portable Audio & Video,NaN,1999,NaN,0
4,0,4,2.0,1999-07-06,Female,Portable Audio & Video,NaN,1999,NaN,0


In [4]:
valeurs_uniques = rdf_all['model_attr'].unique()

print(valeurs_uniques)

['Female' 'Female&Male' 'Male']


In [5]:
rdf_all = rdf_all.dropna(subset=['user_attr'])
rdf_all.reset_index(drop=True, inplace=True)
rdf_all.head()

,item_id,user_id,rating,timestamp,model_attr,category,brand,year,user_attr,split
0,0,28,2.0,1999-12-01,Female,Portable Audio & Video,NaN,1999,Female,0
1,3,81,5.0,2000-03-31,Female,Camera & Photo,NaN,2005,Male,0
2,7,131,4.0,2000-06-15,Female,Home Audio,Philips,2001,Male,0
3,3,139,3.0,2000-06-29,Female,Camera & Photo,NaN,2005,Female,0
4,8,178,1.0,2000-10-19,Female,Accessories & Supplies,NaN,2000,Male,0


In [6]:
rdf = rdf_all[["user_id","item_id", "rating"]]

rdf.head()


,user_id,item_id,rating
0,28,0,2.0
1,81,3,5.0
2,131,7,4.0
3,139,3,3.0
4,178,8,1.0


In [7]:
user_df = rdf_all[["user_id", "user_attr"]]

user_df.shape

(174124, 2)

In [8]:
user_df.head()


,user_id,user_attr
0,28,Female
1,81,Male
2,131,Male
3,139,Female
4,178,Male


In [9]:
user_type_dict = dict()
for u in range(len(user_df)):
    type_str = user_df.at[u, "user_attr"]
    
    #type_list = type_str.split('|')
    type_list = type_str
    user_type_dict[user_df.at[u, 'user_id']] = [type_list]

In [10]:
print(user_type_dict)

{28: ['Female'], 81: ['Male'], 131: ['Male'], 139: ['Female'], 178: ['Male'], 231: ['Male'], 233: ['Female'], 257: ['Female'], 269: ['Male'], 274: ['Male'], 309: ['Male'], 340: ['Male'], 369: ['Female'], 385: ['Male'], 404: ['Male'], 418: ['Female'], 432: ['Female'], 505: ['Male'], 516: ['Male'], 545: ['Male'], 551: ['Male'], 571: ['Female'], 603: ['Male'], 627: ['Female'], 671: ['Male'], 683: ['Female'], 688: ['Female'], 707: ['Male'], 725: ['Female'], 760: ['Male'], 767: ['Male'], 782: ['Female'], 829: ['Male'], 840: ['Female'], 841: ['Female'], 844: ['Female'], 858: ['Male'], 868: ['Female'], 939: ['Male'], 945: ['Male'], 950: ['Male'], 952: ['Female'], 985: ['Female'], 991: ['Male'], 1011: ['Male'], 1078: ['Male'], 1079: ['Male'], 1081: ['Male'], 1115: ['Female'], 1136: ['Female'], 1140: ['Male'], 1147: ['Female'], 1156: ['Male'], 1168: ['Female'], 1188: ['Female'], 1197: ['Male'], 1205: ['Male'], 1222: ['Male'], 1225: ['Male'], 1236: ['Female'], 1289: ['Male'], 1293: ['Male'], 129

In [11]:
item_set = set(rdf['item_id'].unique())
user_set = set(rdf['user_id'].unique())
print('item num = ' + str(len(item_set)))
print('user num = ' + str(len(user_set)))

item num = 8188
user num = 132393


In [12]:
# count the number for each user type and sort
import operator
type_count = dict()
for l in user_type_dict:
    for g in user_type_dict[l]:
        if not g in type_count:
            type_count[g] = 1
        else:
            type_count[g] += 1

type_count_sorted = sorted(type_count.items(), key=operator.itemgetter(1), reverse=True)
type_count_sorted

[('Female', 71043), ('Male', 61350)]

In [13]:
key_type = ['Female', 'Male']

# get the key_type->user_list dict
key_type_user = dict()
for k in key_type:
    key_type_user[k] = list()
for user in user_type_dict:
    for t in user_type_dict[user]:
        if t in key_type:
            key_type_user[t].append(user)

In [14]:
# collect all the users with key types
key_user_set = set()
for types in key_type_user:
    key_user_set |= set(key_type_user[types])

nonkey_user_set = user_set - key_user_set

In [15]:
# remove the non-key type users in rdf
remove_list = []
for user in nonkey_user_set:
    remove_list += rdf.index[rdf['user_id'] == user].values.tolist()

In [16]:
print(len(remove_list))

0


In [17]:
rdf.drop(remove_list, inplace=True)

C:\ProgramData\Anaconda3\lib\site-packages\pandas\core\frame.py:3997: SettingWithCopyWarning: 
A value is trying to be set on a copy of a slice from a DataFrame

See the caveats in the documentation: https://pandas.pydata.org/pandas-docs/stable/user_guide/indexing.html#returning-a-view-versus-a-copy
  errors=errors,


In [18]:
rdf.reset_index(drop=True, inplace=True)
rating_df = copy.copy(rdf)

In [19]:
len(rdf['user_id'].unique())

132393

In [20]:
rdf = copy.copy(rating_df)

In [21]:
# iteratively remove items and users with less than 2 reviews
rdf.reset_index(drop=True, inplace=True)

rdf['user_freq'] = rdf.groupby('user_id')['user_id'].transform('count')
rdf.drop(rdf.index[rdf['user_freq'] <= 2], inplace=True)
rdf.reset_index(drop=True, inplace=True)
rdf['item_freq'] = rdf.groupby('item_id')['item_id'].transform('count')
rdf.drop(rdf.index[rdf['item_freq'] <= 2], inplace=True)
rdf.reset_index(drop=True, inplace=True)
rdf['user_freq'] = rdf.groupby('user_id')['user_id'].transform('count')
rdf.reset_index(drop=True, inplace=True)
rdf['user_id'].value_counts()

30661     34
7605      30
179602    29
16528     29
22606     28
          ..
355003     1
117441     1
389800     1
104481     1
230146     1
Name: user_id, Length: 7882, dtype: int64

In [22]:
item_list = rdf['item_id'].unique()
user_list = rdf['user_id'].unique()
print('item num = ' + str(len(item_list)))
print('user num = ' + str(len(user_list)))

item num = 2289
user num = 7882


In [23]:
# get the user and item str id->int id dict
i = 0
user_id_dict = dict()
for u in user_list:
    if not u in user_id_dict:
        user_id_dict[u] = i
        i += 1
j = 0
item_id_dict = dict()
for i in item_list:
    if not i in item_id_dict:
        item_id_dict[i] = j
        j += 1

In [24]:
print('sparsity: ' + str(len(rdf) * 1.0 / (len(user_list) * len(item_list))))

sparsity: 0.0014978468451600825


In [25]:
# get the df of train, vali, and test set
rdf.reset_index(inplace=True, drop=True)
train_df = rdf.copy()
vali_df = rdf.copy()
test_df = rdf.copy()

train_ratio = 0.8
vali_ratio = 0.0
test_ratio = 0.2
num_all = len(rdf)
vali_idx = []
test_idx = []

test_vali_idx = []
i = 0
num_user = len(user_list)
for u in user_list:
    u_idx = train_df.index[train_df['user_id'] == u]
    idx_len = len(u_idx)
    test_len = int(idx_len * (test_ratio + vali_ratio))
    if test_len == 0:
        test_len = 1
    tmp = np.random.choice(u_idx, size=test_len, replace=False)
    test_vali_idx += tmp.tolist()
    i += 1
    if i % 5000 == 0:
        print(str(i) + '/' + str(num_user))

# tmp = (np.random.choice(range(num_all), size=(test_len+vali_len), replace=False)).tolist()
test_len = int(len(test_vali_idx) * test_ratio / (test_ratio + vali_ratio))
vali_len = int(len(test_vali_idx) - test_len)
test_idx = (np.random.choice(test_vali_idx, size=test_len, replace=False)).tolist()
vali_idx = (np.random.choice(test_vali_idx, size=vali_len, replace=False)).tolist()

test_set = set(test_idx)
vali_set = set(vali_idx)
train_set = set(range(num_all)) - test_set - vali_set
train_idx = list(train_set)
train_df.drop((test_idx + vali_idx), axis=0, inplace=True)
test_df.drop((train_idx + vali_idx), axis=0, inplace=True)
vali_df.drop((train_idx + test_idx), axis=0, inplace=True)

5000/7882


In [26]:
len(train_df['user_id'].unique())

7645

In [27]:
common_users =test_df['user_id'].isin(train_df['user_id'])# ne conserver que les utilisateurs donc l'id est présent dans le train set
test_df = test_df[common_users]
#print(common_users)
common_uid =rdf['user_id'].isin(train_df['user_id'])
rdf = rdf[common_uid]
test_df.reset_index(drop=True, inplace=True)
rdf.reset_index(drop=True, inplace=True)
print(train_df)

       user_id  item_id  rating  user_freq  item_freq
0          683        3     1.0          3          3
2         1156       81     5.0          8         14
4         1188       81     5.0          6         14
5         1188       80     4.0          6          8
7         1156       83     5.0          8          8
...        ...      ...     ...        ...        ...
27016   312530     8341     2.0          2         15
27017    19236     7089     4.0          4         37
27018    56662     9041     5.0          3          3
27019   932318     7089     5.0          3         37
27021   495678     7533     4.0          4          4

[18968 rows x 5 columns]


In [28]:
item_list = rdf['item_id'].unique()
user_list = rdf['user_id'].unique()
print('item num = ' + str(len(item_list)))
print('user num = ' + str(len(user_list)))

item num = 2289
user num = 7645


In [29]:
i = 0
user_id_dict = dict()
for u in user_list:
    if not u in user_id_dict:
        user_id_dict[u] = i
        i += 1
j = 0
item_id_dict = dict()
for i in item_list:
    if not i in item_id_dict:
        item_id_dict[i] = j
        j += 1

In [31]:
# get the matrix of train, vali and test set

train_df.reset_index(drop=True, inplace=True)
test_df.reset_index(drop=True, inplace=True)
vali_df.reset_index(drop=True, inplace=True)
rdf.reset_index(drop=True, inplace=True)
train = np.zeros((len(user_list), len(item_list)))
test = np.zeros((len(user_list), len(item_list)))
vali = np.zeros((len(user_list), len(item_list)))
for r in range(len(train_df)):
    train[user_id_dict[train_df.at[r, 'user_id']], item_id_dict[train_df.at[r, 'item_id']]] = 1.0
for r in range(len(test_df)):
    test[user_id_dict[test_df.at[r, 'user_id']], item_id_dict[test_df.at[r, 'item_id']]] = 1.0
for r in range(len(vali_df)):
    vali[user_id_dict[vali_df.at[r, 'user_id']], item_id_dict[vali_df.at[r, 'item_id']]] = 1.0

In [32]:
print(len(train_df))

18968


In [33]:
# get the user int id-> str id list, and the same for item 
user_list = user_id_dict.keys()
user_idd_list = list()
for u in range(len(user_list)):
    user_idd_list.append('')
for user in user_id_dict:
    user_idd_list[user_id_dict[user]] = user

item_list = item_id_dict.keys()
item_idd_list = list()
for i in range(len(item_list)):
    item_idd_list.append('')
for item in item_id_dict:
    item_idd_list[item_id_dict[item]] = item
    
# get the user int id->types list
user_idd_type_list = list()
for u in range(len(user_idd_list)):
    user_idd_type_list.append(user_type_dict[user_idd_list[u]])

In [34]:
train_df.head()

,user_id,item_id,rating,user_freq,item_freq
0,683,3,1.0,3,3
1,1156,81,5.0,8,14
2,1188,81,5.0,6,14
3,1188,80,4.0,6,8
4,1156,83,5.0,8,8


In [35]:
train_df.drop('user_freq', axis=1, inplace=True)
train_df.drop('item_freq', axis=1, inplace=True)
vali_df.drop('user_freq', axis=1, inplace=True)
vali_df.drop('item_freq', axis=1, inplace=True)
test_df.drop('user_freq', axis=1, inplace=True)
test_df.drop('item_freq', axis=1, inplace=True)
rdf.drop('user_freq', axis=1, inplace=True)
rdf.drop('item_freq', axis=1, inplace=True)

In [36]:
len(test_df['user_id'].unique())

7645

In [37]:
# get df for rdf, train, vali, test with int id for user and item
import copy
rating_df = copy.copy(rdf)
for i in range(len(rdf)):
    rating_df.at[i, 'user_id'] = user_id_dict[rating_df.at[i, 'user_id']]
    rating_df.at[i, 'item_id'] = item_id_dict[rating_df.at[i, 'item_id']]

training_df = copy.copy(train_df)
for i in range(len(training_df)):
    training_df.at[i, 'user_id'] = user_id_dict[training_df.at[i, 'user_id']]
    training_df.at[i, 'item_id'] = item_id_dict[training_df.at[i, 'item_id']]

valiing_df = copy.copy(vali_df)
for i in range(len(valiing_df)):
    valiing_df.at[i, 'user_id'] = user_id_dict[valiing_df.at[i, 'user_id']]
    valiing_df.at[i, 'item_id'] = item_id_dict[valiing_df.at[i, 'item_id']]

testing_df = copy.copy(test_df)
for i in range(len(testing_df)):
    testing_df.at[i, 'user_id'] = user_id_dict[testing_df.at[i, 'user_id']]
    testing_df.at[i, 'item_id'] = item_id_dict[testing_df.at[i, 'item_id']]

In [38]:
# generate the rating list for each key type, get the type->ratings dict
rdf.reset_index(drop=True, inplace=True)
key_type_rating = dict()
for k in key_type:
    key_type_rating[k] = 0.0
for r in range(len(rdf)):
    user = rdf.at[r, 'user_id']
    gl = user_type_dict[user]
    for k in key_type:
        if k in gl:
            key_type_rating[k] += 1.0

# get the user int id->type list
type_user_vector = dict()
for k in key_type:
    type_user_vector[k] = np.zeros((1, len(user_list)))
for u in range(len(user_idd_type_list)):
    type_list = user_idd_type_list[u]
    for t in type_list:
        if t in key_type:
            type_user_vector[t][0,u] = 1.0

In [39]:
len(user_idd_type_list)

7645

In [40]:
with open("user_type_dict_amazon.pkl", "wb") as f:
    pickle.dump(user_type_dict, f)
with open("type_user_vector_amazon.pkl", "wb") as f:
    pickle.dump(type_user_vector, f, pickle.HIGHEST_PROTOCOL)
with open("key_type_amazon.pkl", "wb") as f:
    pickle.dump(key_type, f, pickle.HIGHEST_PROTOCOL)
with open("user_id_dict_amazon.pkl", "wb") as f:
    pickle.dump(user_id_dict, f, pickle.HIGHEST_PROTOCOL)
with open("item_id_dict_amazon.pkl", "wb") as f:
    pickle.dump(item_id_dict, f, pickle.HIGHEST_PROTOCOL)
# with open("rdf.pkl", "wb") as f:
#     pickle.dump(rdf, f, pickle.HIGHEST_PROTOCOL)
with open("rating_df_amazon.pkl", "wb") as f:
    pickle.dump(rating_df, f, pickle.HIGHEST_PROTOCOL)
with open("training_df_amazon.pkl", "wb") as f:
    pickle.dump(training_df, f, pickle.HIGHEST_PROTOCOL)
with open("valiing_df_amazon.pkl", "wb") as f:
    pickle.dump(valiing_df, f, pickle.HIGHEST_PROTOCOL)
with open("testing_df_amazon.pkl", "wb") as f:
    pickle.dump(testing_df, f, pickle.HIGHEST_PROTOCOL)
with open("user_idd_type_list_amazon.pkl", "wb") as f:
    pickle.dump(user_idd_type_list, f, pickle.HIGHEST_PROTOCOL)
with open("item_idd_list_amazon.pkl", "wb") as f:
    pickle.dump(item_idd_list, f, pickle.HIGHEST_PROTOCOL)
with open("user_idd_list_amazon.pkl", "wb") as f:
    pickle.dump(user_idd_list, f, pickle.HIGHEST_PROTOCOL)
with open("key_type_rating_amazon.pkl", "wb") as f:
    pickle.dump(key_type_rating, f, pickle.HIGHEST_PROTOCOL)
    
with open("train_amazon.mat", "wb") as f:
    np.save(f, train)
with open("test_amazon.mat", "wb") as f:
    np.save(f, test)
with open("vali_amazon.mat", "wb") as f:
    np.save(f, vali)

In [41]:
# count the number for each user type and sort
import pickle5 as pickle
from operator import itemgetter
user_list = rdf['user_id'].unique()
#user_type_dict = pickle.load(open('./user_type_dict.pkl'))

with open('./user_type_dict_amazon.pkl', 'rb') as f:
    user_type_dict = pickle.load(f,encoding='latin1')

with open('./key_type_amazon.pkl', 'rb') as f:
    key_type = pickle.load(f,encoding='latin1')

#key_type = pickle.load(open('./key_type.pkl'))

type_count = dict()
for u in user_list:
    gl = user_type_dict[u]
    for g in gl:
        if g in key_type:
            if not g in type_count:
                type_count[g] = 1
            else:
                type_count[g] += 1

# with open("genre_count.pkl", "wb") as f:
#     pickle.dump(genre_count, f, pickle.HIGHEST_PROTOCOL)
                
type_count_sorted = sorted(type_count.items(), key=itemgetter(1), reverse=True)
type_count_sorted

[('Male', 5199), ('Female', 2446)]

In [42]:
with open("type_count_amazon.pkl", "wb") as f:
    pickle.dump(type_count_sorted, f)

In [43]:
import numpy as np

import copy as copy
train = np.load('./train_amazon.mat')

with open('./user_idd_type_list_amazon.pkl', 'rb') as f:
    user_idd_type_list = pickle.load(f,encoding='latin1')

user_idd_type_list = np.array(user_idd_type_list)


mask = 1.0 * (train > 0)
item_type_count = list()
for i in range(train.shape[1]):
    temp_type_count = copy.copy(type_count)
    mask_i = mask[:, i]
    gll = user_idd_type_list[mask_i == 1.0]
    for gl in gll:
        for g in gl:
            if g in key_type:
                temp_type_count[g] -= 1
    item_type_count.append(temp_type_count)
# with open("user_genre_count.pkl", "wb") as f:
#     pickle.dump(user_genre_count, f, pickle.HIGHEST_PROTOCOL)

In [44]:
with open("item_type_count_amazon.pkl", "wb") as f:
    pickle.dump(item_type_count, f)

In [45]:
len(item_type_count)

2289

In [46]:
type_count_sorted

[('Male', 5199), ('Female', 2446)]

In [47]:
with open('./key_type_rating_amazon.pkl', 'rb') as f:
    key_type_rating = pickle.load(f,encoding='latin1')


#key_type_rating = pickle.load(open('./key_genre_rating.pkl'))
type_avg_like = dict()
for k in key_type:
    type_avg_like[k] = key_type_rating[k] * 1.0 / type_count[k]

In [48]:
type_avg_like_sorted = sorted(type_avg_like.items(), key=itemgetter(1), reverse=True)
type_avg_like_sorted

[('Male', 3.529140219272937), ('Female', 3.4501226492232218)]